In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

from agent.helpers.docker import list_containers as list_containers_raw
from agent.helpers.normalize import normalize

load_dotenv()

In [ ]:
@tool
async def docker_status(container: str) -> str:
    """Return the state of a Docker container (read-only): state, status, image.
    Matching is separator-agnostic, e.g. 'agent_api' matches 'marconi-agent-api'."""
    containers = await list_containers_raw()
    needle = normalize(container)
    for c in containers:
        name = (c.get("Names") or ["?"])[0].lstrip("/")
        if needle in normalize(name):
            return f"{name}: state={c.get('State')}, status={c.get('Status')}, image={c.get('Image')}"
    return f"(no container matching '{container}')"

In [ ]:
llm = ChatOpenAI(
    model=os.environ["LLM_MODEL"],
    base_url=os.environ["LLM_BASE_URL"],
    api_key=os.environ.get("LLM_API_KEY"),
    temperature=0,
).bind_tools([docker_status])

In [ ]:
SYSTEM = "You are an infrastructure-operations assistant. Use the docker_status tool when relevant, then answer concisely."

transcript: list = []


async def ask(question: str) -> str:
    """Ask one question. Makes at most one docker_status call, then returns the model's answer."""
    messages = [SystemMessage(SYSTEM), HumanMessage(question)]
    reply = await llm.ainvoke(messages)
    messages.append(reply)
    if reply.tool_calls:
        call = reply.tool_calls[0]
        result = await docker_status.ainvoke(call["args"] or {})
        messages.append(ToolMessage(tool_call_id=call["id"], content=str(result)[:4000]))
        reply = await llm.ainvoke(messages)
        messages.append(reply)
    transcript[:] = messages
    return reply.content

In [ ]:
print(await ask("Is the api container running?"))

In [ ]:
def show_transcript(transcript: list) -> None:
    """Print each message with its tool calls, token usage, and reasoning."""
    for m in transcript:
        label = type(m).__name__
        body = m.content if not isinstance(m, ToolMessage) else f"[tool] {m.content}"
        if not body:
            body = "(tool call only)" if getattr(m, "tool_calls", None) else "(empty)"
        print(f"{label}: {body.replace(chr(10), chr(10) + '  ')}")

        reasoning = m.additional_kwargs.get("reasoning_content")
        if reasoning:
            print(f"  reasoning: {reasoning}")

        for call in getattr(m, "tool_calls", None) or []:
            print(f"  tool_call: {call['name']}({call['args']})")

        usage = getattr(m, "usage_metadata", None)
        if usage:
            print(f"  tokens: in={usage['input_tokens']} out={usage['output_tokens']} total={usage['total_tokens']}")

In [ ]:
show_transcript(transcript)